# M2–M5 — đối chứng lựa chọn 30 tuyến

**M là mã phương pháp, không phải trọng số hoặc số thứ tự bắt buộc.** Có thể chạy M2/M3 độc lập sau khi đã có cùng pool 60 và dữ liệu kiểm tra; M4/M5 được lấy lại từ lịch sử chọn tuyến đã lưu.

Chọn **Python thường** có NumPy và pandas. Notebook chỉ gọi module, không khởi động QGIS, không gửi HTTP và không thay bộ 30 gốc. [Hướng dẫn đầy đủ](../docs/lez/SELECTION_BENCHMARKS.md).


In [ ]:
from pathlib import Path
import importlib
import importlib.util
import json
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "scripts/lez/selection_benchmarks.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Mở thư mục work trong VS Code rồi chạy notebook từ work/notebooks.")
missing = [name for name in ("numpy", "pandas")
           if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError("Kernel thiếu: " + ", ".join(missing)
                       + ". Chọn Python có thư viện trong requirements-notebook.txt.")
scripts = str(ROOT / "scripts")
if scripts not in sys.path:
    sys.path.insert(0, scripts)
bench = importlib.import_module("lez.selection_benchmarks")
if Path(bench.__file__).resolve() != (ROOT / "scripts/lez/selection_benchmarks.py").resolve():
    raise RuntimeError("Kernel đang giữ module từ dự án khác. Restart Kernel rồi chạy lại.")
import pandas as pd
from IPython.display import display
print("Đã tìm thấy work; đang dùng Python thường:", sys.executable)


### 1. Kiểm tra dữ liệu trước khi chạy

Kiểm đầu vào, hash, ma trận 12 ô, quota và lịch sử bộ 30. Thiếu bundle hoặc kiểm tra không đạt thì dừng; không dựng lại GIS hoặc tự tải dữ liệu để lấp chỗ thiếu. Sau khi clone, cần khôi phục dữ liệu như [hướng dẫn máy mới](../docs/setup/GETTING_STARTED.md).


In [ ]:
check = bench.preflight_benchmarks(ROOT)
print(json.dumps(check, ensure_ascii=False, indent=2))


### 2. Chọn các phương pháp cần chạy

- **M2:** mỗi lượt chọn tuyến hợp lệ tạo D thấp nhất; 10 cặp khởi đầu cố định, không hoán đổi.
- **M3:** mỗi lượt chọn tuyến bổ sung nhiều mét đường RC1–RC4 chưa được phủ nhất; cùng 10 khởi đầu xếp theo D để kiểm soát cách bắt đầu, không hoán đổi. Sau khởi đầu, M3 không dùng D để chọn tuyến hoặc chọn lượt chạy tốt nhất.
- **M4:** bộ 30 trước hoán đổi, đọc từ lịch sử chọn C1–C7 đã kiểm chứng.
- **M5:** bộ 30 sau hoán đổi trong cùng lịch sử; chính là bộ gốc được dùng đối chiếu.

Có thể chọn chỉ `["M2"]` hoặc `["M3"]`; thứ tự trong danh sách không quyết định kết quả. **M1 còn chờ chốt quy tắc phân bổ; không đưa M1 vào danh sách.** M0 đã có notebook riêng, không bốc lại ở đây.


In [ ]:
METHODS = ["M2", "M3", "M4", "M5"]


### 3. Chạy hoặc tiếp tục

Các phương pháp dùng cùng quota **A10/B8/C12**, điều kiện tiền khảo sát và trùng hành lang **<70%**. Không ép có sẵn 11/12 ô; số ô là kết quả đánh giá. Ctrl+C giữ kết quả từng phương pháp đã hoàn thành; chạy lại cell để dùng lại cache hợp lệ. Phương pháp chưa hoàn thành có thể phải chạy lại các khởi đầu từ đầu.


In [ ]:
report = bench.run_benchmarks(ROOT, methods=METHODS, progress=print)
print("Trạng thái:", report.get("status"))
print({name: result.get("status") for name, result in report["methods"].items()})


### 4. So sánh bằng cùng mẫu số

D dùng chiều dài theo lượt đi RC1–RC4; độ phủ dùng chiều dài hợp không trùng. Connector chỉ nằm trong tổng km hành trình. D thấp hơn và độ phủ cao hơn là hai mục tiêu khác nhau; xem cả km, trùng lặp và số ô. C6 chưa có bằng chứng vẫn chưa được xác minh.


In [ ]:
rows = []
for method, result in report["methods"].items():
    row = {"method": method, "status": result.get("status")}
    row.update(result.get("metrics", {}))
    rows.append(row)
comparison = pd.json_normalize(rows)
columns = ["method", "status", "D_combined", "D_by_domain.inside", "D_by_domain.outside",
           "traversed_length_m", "unique_rc_length_m", "unique_rc_coverage_fraction",
           "max_unique_corridor_overlap_fraction", "material_cells", "quantile_groups_covered"]
display(comparison[[c for c in columns if c in comparison.columns]])


### 5. Xem 30 mã tuyến và các khởi đầu

Bộ được chọn là kết quả tốt nhất tìm thấy trong các khởi đầu hoàn thành, chưa chứng minh tối ưu toàn cục. Nếu một khởi đầu bế tắc hoặc một phương pháp thất bại, giữ lý do trong kết quả để kiểm tra, không tự nới điều kiện.


In [ ]:
selected = [{"method": method, "status": result.get("status"),
             "selected_route_ids": ", ".join(result.get("selected_route_ids", [])),
             "source": result.get("provenance", {}).get("source"),
             "history_path": result.get("history_path")}
            for method, result in report["methods"].items()]
display(pd.DataFrame(selected))

seed_rows = []
for method, result in report["methods"].items():
    for seed_result in result.get("seed_runs", []):
        seed_rows.append({"method": method, **seed_result})
if seed_rows:
    display(pd.json_normalize(seed_rows))
else:
    print("Lượt chọn này chỉ đọc bộ M4/M5 đã lưu; xem báo cáo nguồn để truy vết khởi đầu.")


### 6. Xem trạng thái và nơi lưu kết quả

Các CSV/JSON và manifest được ghi riêng; không chép kết quả đối chứng đè lên bộ 30 gốc. `benchmark_status` chỉ đọc báo cáo mới nhất. Nếu muốn chạy lại từ đầu kernel sau khi dừng, chạy cell thiết lập, cấu hình rồi cell chạy ở mục 3. Bộ nhớ đệm lưu theo phương pháp đã hoàn thành, không cam kết tiếp tục đúng giữa một lượt chọn.


In [ ]:
for key, value in report["paths"].items():
    print(key, ":", value)
print(json.dumps(bench.benchmark_status(ROOT), ensure_ascii=False, indent=2))


### Giới hạn diễn giải

M2/M3 dùng dữ liệu và gate tiền khảo sát đã lưu; M4/M5 được trích từ lịch sử cùng phiên bản. A×RC1 vẫn nằm trong mạng tham chiếu dù pool không tạo được tuyến đại diện. Không suy ra điều kiện thực địa, tính pháp lý của đa giác ranh giới hoặc chất lượng chu trình lái từ bảng này.

M0 ở [notebook 10](10_random_baseline.ipynb); biểu đồ M0 ở [notebook 11](11_random_comparison_plots.ipynb). M1 chưa được triển khai vì quy tắc chọn theo tỷ trọng chưa chốt.
